In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## First Look at the Data

Before doing anything else, we check the basic structure of the data:
how many rows and columns we have, what type each column is and how
many values are missing in each column. This helps us understand what
needs to be cleaned before we can use this data for modeling

In [99]:
df = pd.read_csv('data/movies.csv' , encoding='latin-1')
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [100]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  str    
 1   Year      14981 non-null  str    
 2   Duration  7240 non-null   str    
 3   Genre     13632 non-null  str    
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   str    
 6   Director  14984 non-null  str    
 7   Actor 1   13892 non-null  str    
 8   Actor 2   13125 non-null  str    
 9   Actor 3   12365 non-null  str    
dtypes: float64(1), str(9)
memory usage: 1.2 MB


In [101]:
df.shape

(15509, 10)

In [102]:
df.isnull().sum()

Name           0
Year         528
Duration    8269
Genre       1877
Rating      7590
Votes       7589
Director     525
Actor 1     1617
Actor 2     2384
Actor 3     3144
dtype: int64

The year and votes column look like numbers but stored as text where year is sourrounded by brackets and 
votes ara comma separated

In [103]:
df[['Year', 'Votes']].head(10)

,Year,Votes
0,NaN,NaN
1,(2019),8
2,(2021),NaN
3,(2019),35
4,(2010),NaN
5,(1997),827
6,(2005),"1,086"
7,(2008),NaN
8,(2012),326
9,(2014),11


## Cleaing year column 
Removing brackets from year column and converting it to numbers
we use `errors='coerce'`, which turns any value that cannot be converted into a missing value instead of stopping the whole process with an error

In [104]:
df['Year'] =df['Year'].str.replace('(', '', regex=False)
df['Year']= df['Year'].str.replace(')', '', regex=False)
df['Year'] =pd.to_numeric(df['Year'], errors='coerce')
df['Year'].head(10)

0       NaN
1    2019.0
2    2021.0
3    2019.0
4    2010.0
5    1997.0
6    2005.0
7    2008.0
8    2012.0
9    2014.0
Name: Year, dtype: float64

## Cleaning votes column
Removing commas from large numbers and managing errors as same as year

In [105]:
df['Votes']=df['Votes'].str.replace(',', '', regex=False)
df['Votes']= pd.to_numeric(df['Votes'],errors='coerce')
df['Votes'].head(10)

0       NaN
1       8.0
2       NaN
3      35.0
4       NaN
5     827.0
6    1086.0
7       NaN
8     326.0
9      11.0
Name: Votes, dtype: float64

## Checking and cleaning the durarion column
The Duration column has the word "min" attached to every value, like
"109 min". We remove this text and convert the column into a proper
number representing minutes

In [106]:
df['Duration'].head(10)

0        NaN
1    109 min
2     90 min
3    110 min
4    105 min
5    147 min
6    142 min
7     59 min
8     82 min
9    116 min
Name: Duration, dtype: str

In [107]:
df['Duration'] = df['Duration'].str.replace(' min', '', regex=False)
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')
df['Duration'].head(10)

0      NaN
1    109.0
2     90.0
3    110.0
4    105.0
5    147.0
6    142.0
7     59.0
8     82.0
9    116.0
Name: Duration, dtype: float64

## Handling Missing Values

Now that our numeric columns are properly cleaned, we need to decide
what to do with the missing values in each column. Different columns
need different treatment depending on how important they are and how
much data is missing.

Our plan:
- Rating: This is what we are trying to predict, so any row missing
  a Rating is useless for training. We will drop these rows entirely.
- Votes: Almost always missing exactly when Rating is missing, so
  these rows get dropped along with Rating anyway.
- Duration: Over half the values are missing. We will fill missing
  values with the median duration, since the median is not affected
  much by extreme outliers.
- Year: Only a small number missing. We will fill these with the
  median year.
- Genre, Director, Actor 1/2/3: These are text columns. We will fill
  missing values with the label "Unknown" instead of guessing a name.

In [108]:
df = df.dropna(subset=['Rating']) #Dropping rating with missing values
df.shape

(7919, 10)

## Checking Votes Column

We already dropped rows where Rating was missing. Votes was missing
in almost the same rows as Rating. Let us check how many missing
values are left in Votes now.

In [109]:
df['Votes'].isnull().sum()

np.int64(0)

## Filling Missing Values

Duration and Year are numbers. We fill missing values with the median.
Median is not affected much by very high or very low values.

Genre Director Actor 1 Actor 2 Actor 3 are text columns. We cannot
average a name or a genre. So we fill missing values with the word
"Unknown".

In [110]:
df['Duration'] = df['Duration'].fillna(df['Duration'].median())
df['Year'] = df['Year'].fillna(df['Year'].median())

df['Genre'] = df['Genre'].fillna('Unknown')
df['Director'] = df['Director'].fillna('Unknown')
df['Actor 1'] = df['Actor 1'].fillna('Unknown')
df['Actor 2'] = df['Actor 2'].fillna('Unknown')
df['Actor 3'] = df['Actor 3'].fillna('Unknown')

df.isnull().sum()

Name        0
Year        0
Duration    0
Genre       0
Rating      0
Votes       0
Director    0
Actor 1     0
Actor 2     0
Actor 3     0
dtype: int64

## Final Check After Cleaning

Let us check the dataset one more time. This confirms every column
has no missing values left, except Name and Year which we already
handled, and that all columns have the correct data type.

In [111]:
df.isnull().sum()

Name        0
Year        0
Duration    0
Genre       0
Rating      0
Votes       0
Director    0
Actor 1     0
Actor 2     0
Actor 3     0
dtype: int64

In [112]:
df.info()

<class 'pandas.DataFrame'>
Index: 7919 entries, 1 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      7919 non-null   str    
 1   Year      7919 non-null   float64
 2   Duration  7919 non-null   float64
 3   Genre     7919 non-null   str    
 4   Rating    7919 non-null   float64
 5   Votes     7919 non-null   float64
 6   Director  7919 non-null   str    
 7   Actor 1   7919 non-null   str    
 8   Actor 2   7919 non-null   str    
 9   Actor 3   7919 non-null   str    
dtypes: float64(4), str(6)
memory usage: 680.5 KB


## Dropping the Name Column

Name does not help predict rating. It is just a label for each movie.
We remove it before building the model. We keep it out of the model
input but the original column stays in the file if needed later.

In [113]:
df = df.drop('Name', axis=1)
df.head()

,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
1,2019.0,109.0,Drama,7.0,8.0,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
3,2019.0,110.0,"Comedy, Romance",4.4,35.0,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
5,1997.0,147.0,"Comedy, Drama, Musical",4.7,827.0,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor
6,2005.0,142.0,"Drama, Romance, War",7.4,1086.0,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma
8,2012.0,82.0,"Horror, Mystery, Thriller",5.6,326.0,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia


In [114]:
df.columns

Index(['Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director', 'Actor 1',
       'Actor 2', 'Actor 3'],
      dtype='str')